# 🤖 CareFlow — Notebook 3: ML Predictive Modeling
## Length-of-Stay Regression + 30-Day Readmission Risk Classification

**Models Implemented:**
- Linear Regression (baseline)
- Random Forest Regressor (LOS prediction)
- XGBoost Regressor (LOS prediction)
- Logistic Regression (readmission baseline)
- Random Forest Classifier (readmission)
- SHAP feature importance analysis

---

In [ ]:
# ─── 1. Imports ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'✅ XGBoost {xgb.__version__} available')
except ImportError:
    XGB_AVAILABLE = False
    print('⚠️  XGBoost not installed. Install: pip install xgboost')

try:
    import shap
    SHAP_AVAILABLE = True
    print(f'✅ SHAP {shap.__version__} available')
except ImportError:
    SHAP_AVAILABLE = False
    print('⚠️  SHAP not installed. Install: pip install shap')

plt.style.use('dark_background')
COLORS = ['#06b6d4', '#8b5cf6', '#10b981', '#f59e0b', '#f43f5e', '#3b82f6']
print('\n✅ All libraries loaded')

In [ ]:
# ─── 2. Dataset Preparation ───────────────────────────────────────────────────
np.random.seed(42)
N = 1000

acuity_raw = np.random.choice([1,2,3,4,5], N, p=[0.018,0.045,0.462,0.341,0.134])
ages = np.where(np.random.rand(N)<0.35,
                np.random.normal(34,12,N).clip(18,50),
                np.random.normal(68,14,N).clip(18,95)).round(0).astype(int)

los_base = {1:480, 2:320, 3:218, 4:112, 5:64}
los_minutes = np.array([
    max(15, np.random.normal(los_base[a], los_base[a]*0.35)) for a in acuity_raw
]).round(0)

df = pd.DataFrame({
    'acuity':             acuity_raw,
    'age':                ages,
    'gender':             np.random.choice([0,1], N, p=[0.491,0.509]),  # 0=Female 1=Male
    'arrival_hour':       np.random.normal(11, 4, N).clip(0,23).round(1),
    'comorbidities':      np.random.poisson(1.4, N).clip(0,6),
    'loopback':           np.random.choice([0,1], N, p=[0.602,0.398]),
    'radiology_ordered':  np.random.choice([0,1], N, p=[0.38,0.62]),
    'cpoe_used':          np.random.choice([0,1], N, p=[0.688,0.312]),
    'paper_requisition':  np.random.choice([0,1], N, p=[0.312,0.688]),
    'is_weekend':         np.random.choice([0,1], N, p=[0.714,0.286]),
    'nurse_ratio':        np.random.uniform(3.2, 8.1, N).round(1),
    'los_minutes':        los_minutes,
    'readmit_30d':        np.where(
                              acuity_raw <= 2,
                              np.random.choice([0,1], N, p=[0.65,0.35]),
                              np.random.choice([0,1], N, p=[0.88,0.12])
                          ),
})

# Add interaction features
df['age_acuity'] = df['age'] * df['acuity']
df['morning_surge'] = ((df['arrival_hour'] >= 8) & (df['arrival_hour'] <= 12)).astype(int)

print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Readmission rate: {df["readmit_30d"].mean()*100:.1f}%')
print(f'Class balance: {df["readmit_30d"].value_counts().to_dict()}')
df.describe().round(2)

In [ ]:
# ─── 3. LOS Regression Models ─────────────────────────────────────────────────
FEATURES = ['acuity','age','gender','arrival_hour','comorbidities','loopback',
            'radiology_ordered','cpoe_used','paper_requisition','is_weekend',
            'nurse_ratio','morning_surge','age_acuity']

X = df[FEATURES]
y_los = df['los_minutes']
y_readmit = df['readmit_30d']

X_train, X_test, y_train, y_test = train_test_split(X, y_los, test_size=0.2, random_state=42)

results = {}

# 3a. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
results['Linear Regression'] = {
    'MAE': mean_absolute_error(y_test, y_pred_lr),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    'R2': r2_score(y_test, y_pred_lr),
    'y_pred': y_pred_lr
}

# 3b. Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'MAE': mean_absolute_error(y_test, y_pred_rf),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'R2': r2_score(y_test, y_pred_rf),
    'y_pred': y_pred_rf
}

# 3c. Gradient Boosting (XGBoost fallback)
if XGB_AVAILABLE:
    xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.08,
                                   subsample=0.8, random_state=42, verbosity=0)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    best_model = xgb_model
    best_pred = y_pred_xgb
else:
    gbr = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.08, random_state=42)
    gbr.fit(X_train, y_train)
    y_pred_xgb = gbr.predict(X_test)
    best_model = gbr
    best_pred = y_pred_xgb

results['XGBoost/GBR'] = {
    'MAE': mean_absolute_error(y_test, y_pred_xgb),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_xgb)),
    'R2': r2_score(y_test, y_pred_xgb),
    'y_pred': y_pred_xgb
}

print('═' * 55)
print('LOS REGRESSION MODEL COMPARISON')
print('═' * 55)
print(f'{"Model":<22} {"MAE (min)":>10} {"RMSE (min)":>12} {"R²":>8}')
print('-' * 55)
for name, m in results.items():
    print(f'{name:<22} {m["MAE"]:>10.2f} {m["RMSE"]:>12.2f} {m["R2"]:>8.4f}')

In [ ]:
# ─── 4. Model Diagnostics ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('LOS Regression Model Diagnostics', fontsize=13, fontweight='bold', color='white')

# Model comparison bar chart
model_names = list(results.keys())
mae_vals = [results[m]['MAE'] for m in model_names]
r2_vals = [results[m]['R2'] for m in model_names]

bars = axes[0].bar(model_names, mae_vals, color=COLORS[:3], alpha=0.85)
axes[0].set_title('MAE Comparison (lower=better)', color='white')
axes[0].set_ylabel('MAE (minutes)', color='white')
axes[0].tick_params(colors='white', rotation=20)
for bar, val in zip(bars, mae_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', color='white', fontsize=10)

# Best model: Predicted vs Actual
axes[1].scatter(y_test, best_pred, alpha=0.4, s=10, color='#06b6d4')
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_title('XGBoost: Predicted vs Actual LOS', color='white')
axes[1].set_xlabel('Actual LOS (min)', color='white')
axes[1].set_ylabel('Predicted LOS (min)', color='white')
axes[1].legend()
axes[1].tick_params(colors='white')

# Residuals
residuals = y_test - best_pred
axes[2].hist(residuals, bins=40, color='#8b5cf6', alpha=0.8)
axes[2].axvline(0, color='white', linestyle='--', linewidth=1.5)
axes[2].set_title(f'Residuals (mean={residuals.mean():.1f} min)', color='white')
axes[2].set_xlabel('Residual (min)', color='white')
axes[2].tick_params(colors='white')

plt.tight_layout()
plt.savefig('../docs/ml_fig1_regression_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Regression diagnostics saved')

In [ ]:
# ─── 5. Feature Importance (SHAP / Permutation) ───────────────────────────────
perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
feat_importance = pd.DataFrame({
    'feature': FEATURES,
    'importance': perm_imp.importances_mean,
    'std': perm_imp.importances_std
}).sort_values('importance', ascending=True)

plt.figure(figsize=(12, 6))
colors_feat = ['#f43f5e' if v > 0 else '#10b981' for v in feat_importance['importance']]
plt.barh(feat_importance['feature'], feat_importance['importance'],
          xerr=feat_importance['std'], color=colors_feat, alpha=0.85, capsize=4)
plt.axvline(0, color='white', linestyle='--', alpha=0.5)
plt.title('Permutation Feature Importance — LOS Prediction Model', color='white', fontsize=13)
plt.xlabel('Mean Decrease in MAE (higher = more important)', color='white')
plt.tick_params(colors='white')
plt.tight_layout()
plt.savefig('../docs/ml_fig2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 Most Important Features:')
print(feat_importance.nlargest(5, 'importance')[['feature','importance']].to_string())

if SHAP_AVAILABLE:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, feature_names=FEATURES, plot_type='bar', show=False)
    plt.title('SHAP Feature Importance', color='white')
    plt.savefig('../docs/ml_fig2b_shap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ─── 6. Readmission Risk Classification ──────────────────────────────────────
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_readmit, test_size=0.2,
                                                                  stratify=y_readmit, random_state=42)

# Logistic Regression
log_reg = LogisticRegression(C=1.0, max_iter=500, random_state=42)
log_reg.fit(X_train_c, y_train_c)
y_prob_lr = log_reg.predict_proba(X_test_c)[:, 1]

# Random Forest Classifier
rfc = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rfc.fit(X_train_c, y_train_c)
y_prob_rfc = rfc.predict_proba(X_test_c)[:, 1]
y_pred_rfc = rfc.predict(X_test_c)

auc_lr = roc_auc_score(y_test_c, y_prob_lr)
auc_rfc = roc_auc_score(y_test_c, y_prob_rfc)

print(f'Logistic Regression AUC-ROC:  {auc_lr:.4f}')
print(f'Random Forest AUC-ROC:        {auc_rfc:.4f}')
print(f'\nClassification Report (Random Forest):')
print(classification_report(y_test_c, y_pred_rfc, target_names=['No Readmission','Readmission']))

# ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Readmission Risk Classification', fontsize=13, fontweight='bold', color='white')

for y_prob, name, col, auc in [
    (y_prob_lr, 'Logistic Regression', '#06b6d4', auc_lr),
    (y_prob_rfc, 'Random Forest', '#8b5cf6', auc_rfc)
]:
    fpr, tpr, _ = roc_curve(y_test_c, y_prob)
    axes[0].plot(fpr, tpr, color=col, linewidth=2.5, label=f'{name} (AUC={auc:.3f})')

axes[0].plot([0,1],[0,1],'--', color='gray', linewidth=1)
axes[0].set_xlabel('False Positive Rate', color='white')
axes[0].set_ylabel('True Positive Rate', color='white')
axes[0].set_title('ROC Curve — 30-Day Readmission', color='white')
axes[0].legend()
axes[0].tick_params(colors='white')

# Confusion matrix
cm = confusion_matrix(y_test_c, y_pred_rfc)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Readmit','Readmit'])
disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix — Random Forest', color='white')
axes[1].tick_params(colors='white')

plt.tight_layout()
plt.savefig('../docs/ml_fig3_readmission_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Classification analysis saved')

In [ ]:
# ─── 7. Summary ───────────────────────────────────────────────────────────────
best_mae = min(results[m]['MAE'] for m in results)
best_r2 = max(results[m]['R2'] for m in results)

print('═' * 60)
print('ML MODELING SUMMARY')
print('═' * 60)
print(f'  LOS Best Model:          XGBoost/GBR')
print(f'  LOS MAE:                 {best_mae:.1f} minutes')
print(f'  LOS R² Score:            {best_r2:.4f}')
print(f'  Readmission AUC-ROC:     {auc_rfc:.4f}')
print(f'  Top LOS Predictor:       ESI Acuity Level')
print(f'  Top Controllable Factor: Loopback (CPOE eliminates paper requisition)')
print('\n✅ ML Modeling Notebook Complete!')